<h1>DOCUMET LOADER</H1>

In [15]:
from langchain_community.document_loaders import Docx2txtLoader
import os


def document_loader(folder_path):

    document = []

    for filename in os.listdir(folder_path):

        if filename.endswith('.docx'):
            file_path = os.path.join(folder_path,filename)

            loader = Docx2txtLoader(file_path)
            pages = loader.load()

            document.extend(pages)

    return document

In [21]:
document = document_loader('E:\GEN-AI-PROJECTS')

<H1>TEXT SPLITTER</H1>

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_splitter(document):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 50
    )

    chunk = text_splitter.split_documents(document)

    return chunk

In [24]:
chunk = text_splitter(document)

In [25]:
print('the ;ength f the chunks is: ',len(chunk))

the ;ength f the chunks is:  58


<h1>EMBEDDING AND VECTOR DATA BASE</H1|>

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

def create_vector_db(chunk):   

    embedding = HuggingFaceEmbeddings(
        model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    )

    vector_db = FAISS.from_documents(   # semantic  search
        chunk,
        embedding
    )

    return vector_db

<h1>KEYWORD SEARCH</h1>

In [58]:
from rank_bm25 import BM25Okapi

class KEYWORD_SEARCH:
    def __init__(self,chunk):
        self.chunk = chunk

        tokenized_data = []

        for x in chunk:
            token = x.page_content.lower().split()
            tokenized_data.append(token)

        self.bm25 = BM25Okapi(tokenized_data)


    def search(self,query,k=5):

        query = query.lower().split()

        scores = self.bm25.get_scores(query)


        ranked_answers = sorted(range(len(scores)),
                                key = lambda index:scores[index],
                                reverse = True)
        top_indices = ranked_answers[:k]

        results = []

        for s in top_indices:
            results.append(s)
        return results


<h2>hybrid_retriever</h2>

In [59]:
class hybridRetriver:
    def __init__(self,vector_db,bm25):
            self.vector_db = vector_db
            self.bm25 = bm25


    def search(self,query,k=5):
        vector_db_retriver = self.vector_db.vector_store.similarity(query,k=5)
        bm25_retirver = self.bm25.retrive_search(query,k=5)

        combined_result = vector_db_retriver + bm25_retirver

        unique_document = []
        seen_content = set()

        for x in combined_result:
                if x.page_content not in seen_content:

                      unique_document.append(x)

                      seen_content.add(x)
        return unique_document
        